In [0]:

# CREAR ESQUEMA GOLD

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS rendimiento_estudiantil.gold
""")

Leer tablas silver

In [0]:
silver_estudiantes = spark.table(f"rendimiento_estudiantil.silver.estudiantes")

silver_cursos = spark.table(f"rendimiento_estudiantil.silver.cursos")

silver_matriculas = spark.table(f"rendimiento_estudiantil.silver.matriculas")

silver_semestres = spark.table(f"rendimiento_estudiantil.silver.semestres")

## Construcción de la Dimensión Estudiantes

Esta dimensión almacena la información descriptiva de cada estudiante.

In [0]:
dim_estudiantes = (
    silver_estudiantes
    .select(
        "Student_ID",
        "Age",
        "Gender",
        "Region_Type",
        "Family_Size",
        "Parent_Education",
        "Family_Income_Level",
        "University_Name",
        "Home_City",
        "Major_Subject",
        "email"
    )
    .dropDuplicates()
)

In [0]:
(
    dim_estudiantes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_estudiantes")
)

## Construcción de la Dimensión Cursos

Contiene la información académica de cada curso.

In [0]:
dim_cursos = (
    silver_cursos
    .select(
        "id_curso",
        "nombre_curso",
        "major_subject",
        "creditos",
        "nivel"
    )
    .dropDuplicates()
)

In [0]:
(
    dim_cursos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_cursos")
)

## Construcción de la Dimensión Semestres

Esta dimensión representa el calendario académico.

In [0]:
dim_semestres = (
    silver_semestres
    .dropDuplicates()
)

In [0]:
(
    dim_semestres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"rendimiento_estudiantil.gold.dim_semestres")
)

## Construcción de la Tabla de Hechos

La tabla de hechos registra cada matrícula realizada por un estudiante y contiene las métricas principales para el análisis.

In [0]:
fact_rendimiento_academico = (
    silver_matriculas
    .select(
        "id_matricula",
        "Student_ID",
        "Semester_ID",
        "id_curso",
        "fecha_matricula",
        "nota_final",
        "estado",
        "costo_matricula"
    )
)

fact_rendimiento_academico.write \
.mode("overwrite") \
.saveAsTable(f"rendimiento_estudiantil.gold.fact_rendimiento_academico")

##Crear Vista 1
Monitoreo General



In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_monitoreo_riesgo AS

SELECT

f.id_matricula,

f.Student_ID,

e.Gender,

e.Age,

e.Region_Type,

e.Family_Income_Level,

e.Major_Subject,

c.nombre_curso,

s.periodo,

s.anio,

f.nota_final,

f.estado,

CASE

WHEN f.nota_final < 60 THEN 'Alto'

WHEN f.nota_final < 80 THEN 'Medio'

ELSE 'Bajo'

END AS Nivel_Riesgo

FROM rendimiento_estudiantil.gold.fact_rendimiento_academico f

INNER JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON f.Student_ID=e.Student_ID

INNER JOIN rendimiento_estudiantil.gold.dim_cursos c

ON f.id_curso=c.id_curso

INNER JOIN rendimiento_estudiantil.gold.dim_semestres s

ON f.Semester_ID=s.Semester_ID;

##Crear Vista 2
Factores asociados

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_analisis_factores AS

SELECT

e.Gender,

e.Region_Type,

e.Family_Income_Level,

e.Parent_Education,

e.Major_Subject,

AVG(f.nota_final) AS Promedio,

COUNT(*) AS Total_Matriculas,

SUM(

CASE

WHEN f.nota_final<60 THEN 1

ELSE 0

END

) AS Estudiantes_Riesgo

FROM rendimiento_estudiantil.gold.fact_rendimiento_academico f

INNER JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY

e.Gender,

e.Region_Type,

e.Family_Income_Level,

e.Parent_Education,

e.Major_Subject;

##Crear Vista 3
Intervención temprana

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_intervencion_temprana AS

SELECT

f.Student_ID,

e.Gender,

e.Major_Subject,

c.nombre_curso,

s.periodo,

f.nota_final,

CASE

WHEN f.nota_final<60 THEN 'Alto'

WHEN f.nota_final<75 THEN 'Medio'

ELSE 'Bajo'

END AS Nivel_Riesgo

FROM rendimiento_estudiantil.gold.fact_rendimiento_academico f

INNER JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON f.Student_ID=e.Student_ID

INNER JOIN rendimiento_estudiantil.gold.dim_cursos c

ON f.id_curso=c.id_curso

INNER JOIN rendimiento_estudiantil.gold.dim_semestres s

ON f.Semester_ID=s.Semester_ID

ORDER BY

f.nota_final ASC;

## Crear vista 4

Riesgo estudiantes

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_riesgo_estudiantes AS

WITH promedio_estudiante AS (

    SELECT

        Student_ID,

        ROUND(AVG(nota_final),2) AS Promedio_Final,

        COUNT(id_matricula) AS Total_Matriculas,

        SUM(costo_matricula) AS Costo_Total

    FROM rendimiento_estudiantil.gold.fact_rendimiento_academico

    GROUP BY Student_ID

)

SELECT

    p.Student_ID,

    e.Gender,

    e.Age,

    e.Region_Type,

    e.Family_Size,

    e.Parent_Education,

    e.Family_Income_Level,

    e.University_Name,

    e.Home_City,

    e.Major_Subject,

    p.Total_Matriculas,

    p.Costo_Total,

    p.Promedio_Final,

    CASE

        WHEN p.Promedio_Final < 60 THEN 'Alto'

        WHEN p.Promedio_Final < 80 THEN 'Medio'

        ELSE 'Bajo'

    END AS Nivel_Riesgo

FROM promedio_estudiante p

INNER JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON p.Student_ID = e.Student_ID;

## Validación del modelo

In [0]:
display(
    spark.table(f"rendimiento_estudiantil.gold.dim_estudiantes")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.dim_cursos")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.dim_semestres")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.fact_rendimiento_academico")
)

## Validación de las vistas

In [0]:
display(
    spark.table(f"rendimiento_estudiantil.gold.vw_monitoreo_riesgo")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.vw_analisis_factores")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.vw_intervencion_temprana")
)

display(
    spark.table(f"rendimiento_estudiantil.gold.vw_riesgo_estudiantes")
)